In [8]:
# This notebook applies the following basic Machine Learning models:
# Logistic Regression, SVM, KNN and Decision Trees
###

# 0. Preparation
###
# Importing libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

from sklearn.metrics import classification_report, f1_score
from sklearn import linear_model, preprocessing
from sklearn.model_selection import train_test_split
from sklearn import svm, neighbors

In [7]:
pip install zarr

In [2]:
# Mounting GoogleDrive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
# Reading data file from GoogleDrive
df = pd.read_pickle("/content/drive/My Drive/Data Science/Team Project X-Rays/Dataframes/df_basic_1.4.pkl")
df.head()

# Define data name
df_name = "Baseline 1.4"

# Define random subsample for computation efficiency
#df = df.sample(500)


In [4]:
# Explore Data
###

# Check data type is DataFrame
print(type(df))

# Show dimensions
print(df.shape)

# Show labels
print(df.Case.value_counts())

# Check distributions after normalisation
df.describe()

# No scaling applied yet, will be done after train test split

<class 'pandas.core.frame.DataFrame'>
(21105, 4098)
Case
Normal             10191
Lung_Opacity        6012
COVID               3564
Viral Pneumonia     1338
Name: count, dtype: int64


,PX_1,PX_2,PX_3,PX_4,PX_5,PX_6,PX_7,PX_8,PX_9,PX_10,...,PX_4087,PX_4088,PX_4089,PX_4090,PX_4091,PX_4092,PX_4093,PX_4094,PX_4095,PX_4096
count,21105.000000,21105.000000,21105.000000,21105.000000,21105.000000,21105.000000,21105.000000,21105.000000,21105.000000,21105.000000,...,21105.000000,21105.000000,21105.000000,21105.000000,21105.000000,21105.000000,21105.000000,21105.000000,21105.000000,21105.000000
mean,0.000966,0.001104,0.001022,0.001024,0.001133,0.001066,0.001123,0.001060,0.001049,0.001136,...,-0.000539,-0.000617,-0.000505,-0.000494,-0.000539,-0.000547,-0.000572,-0.000578,-0.000458,-0.001796
std,1.000715,1.000839,1.000883,1.000659,1.000745,1.000582,1.000579,1.000479,1.000384,1.000567,...,1.000112,1.000057,0.999906,0.999772,0.999759,0.999768,0.999699,0.999677,0.999890,0.998833
min,-0.695368,-0.631396,-0.638616,-0.660304,-0.678508,-0.696735,-0.708352,-0.716076,-0.725822,-0.734141,...,-2.154784,-1.914149,-1.682842,-1.475565,-1.285164,-1.116746,-0.985124,-0.881995,-0.820450,-0.834223
25%,-0.662774,-0.612610,-0.619492,-0.622123,-0.640521,-0.658996,-0.670988,-0.679055,-0.689137,-0.697829,...,-0.476066,-0.615764,-0.847390,-1.062733,-1.081052,-1.001676,-0.906848,-0.841519,-0.792467,-0.806656
50%,-0.499804,-0.462322,-0.485624,-0.488490,-0.488571,-0.508042,-0.502845,-0.512460,-0.505713,-0.516270,...,0.261992,0.254431,0.226762,0.175762,0.041564,-0.183398,-0.411101,-0.531203,-0.540623,-0.517197
75%,0.249857,0.101259,0.126342,0.198766,0.271176,0.341074,0.375230,0.394555,0.429748,0.427838,...,0.739558,0.765497,0.810252,0.846613,0.883526,0.890591,0.854359,0.764030,0.648638,0.640640
max,3.460364,4.159040,4.237990,4.207763,4.164879,4.114923,4.055674,4.004109,3.951487,3.895618,...,1.535502,1.608066,1.698749,1.814187,1.967871,2.143578,2.341600,2.558467,2.747335,2.680639


In [ ]:
# Check missing values
print(df.info())

print("Missing vars in columns:\n", df.isna().sum())
print("Number of total missing vars:", df.isna().sum().sum())
print("Number of total missing vars (% of all obs):", (df.isna().sum().sum())/(df.shape[0]*df.shape[1]))

# No missing vars. We can continue the ML modelling.

<class 'pandas.core.frame.DataFrame'>
Index: 21105 entries, 0 to 21164
Columns: 4098 entries, Name to PX_4096
dtypes: float64(4096), object(2)
memory usage: 660.0+ MB
None
Missing vars in columns:
 Name       0
Case       0
PX_1       0
PX_2       0
PX_3       0
          ..
PX_4092    0
PX_4093    0
PX_4094    0
PX_4095    0
PX_4096    0
Length: 4098, dtype: int64
Number of total missing vars: 0
Number of total missing vars (% of all obs): 0.0


In [ ]:
# 1. Data preprocessing
###

# Create categorical variable from Case
df["Case"] = df.Case.replace({"Normal": 0, "COVID": 1, "Lung_Opacity": 2, "Viral Pneumonia": 3})
df.Case.astype(int)

# Check construction
print(df.Case.value_counts())

# Split data into target and features
target = df.Case

# Features data: Drop Names and target
data = df.drop(["Name", "Case"], axis = 1)
data.head()
data.shape

# Split data into training and Test sets, save random state
X_train, X_test, y_train, y_test = train_test_split(data, target, test_size = 0.2, random_state = 123)

Case
0    10191
2     6012
1     3564
3     1338
Name: count, dtype: int64


In [ ]:
# 2. Model 1 - Logistic Regression
###
import time
start_time = time.time()

# Instantiate Logistic regression for classification
clf1_name = "Logistic Regression"
clf1 = linear_model.LogisticRegression(solver='lbfgs', C = 1.0, max_iter = 10000)

# Train the model on training data
clf1.fit(X_train, y_train)

# Make predictions on test set
y_pred = clf1.predict(X_test)

# Calc accuracys
clf1_score = clf1.score(X_test, y_test)

# Calc mean unweighted F1 Score in all classes
clf1_f1 = f1_score(y_test, y_pred, average = "macro")

# Measure time
model1_time = (time.time() - start_time)/60

In [ ]:
# Show Results
###

# Modelling Time
print("Model 1: --- %s minutes ---" % model1_time)

# Score and F1-Score
print("The score is:", clf1_score)
print("The mean F1-Score (unweighted) is:", clf1_f1)

# Show Confusion Matrix
cm1 = pd.crosstab(y_test, y_pred, rownames = ['Realised Class'], colnames = ['Predicted Class'])
display(cm1)

# Show classification report
model1_cr = classification_report(y_test, y_pred)
print(model1_cr)


In [ ]:
# 3. Model 2 - linear SVM
###
import time
start_time = time.time()

# Instantiate SVM
clf2_name = "linear SVM"
clf2 = svm.SVC(gamma = 0.01, kernel = "poly")

# Train the model on training data
clf2.fit(X_train, y_train)

# Make predictions on test set
y_pred = clf2.predict(X_test)

# Calc accuracy
clf2_score = clf2.score(X_test, y_test)

# Calc mean unweighted F1 Score in all classes
clf2_f1 = f1_score(y_test, y_pred, average = "macro")

# Measure time
model2_time = (time.time() - start_time)/60

In [ ]:
# Show Results
###

# Modelling Time
print("Model 2: --- %s minutes ---" % model2_time)

# Score and F1-Score
print("The score is:", clf2_score)
print("The mean F1-Score (unweighted) is:", clf2_f1)

# Show Confusion Matrix
cm2 = pd.crosstab(y_test, y_pred, rownames = ['Realised Class'], colnames = ['Predicted Class'])
display(cm2)

# Show classification report
model2_cr = classification_report(y_test, y_pred)
print(model2_cr)

In [ ]:
# 3. Model 3 - KNN
###
from sklearn import neighbors
import time
start_time = time.time()

# Instantiate classifier
clf3_name = "KNN"
clf3 = neighbors.KNeighborsClassifier(n_neighbors = 7, metric = 'minkowski')

# Train the model on training data
clf3.fit(X_train, y_train)

# Make predictions on test set
y_pred = clf3.predict(X_test)

# Calc accuracys
clf3_score = clf3.score(X_test, y_test)

# Calc mean unweighted F1 Score in all classes
clf3_f1 = f1_score(y_test, y_pred, average = "macro")

# Measure time
model3_time = (time.time() - start_time)/60

In [ ]:
# Show Results
###

# Modelling Time
print("Model 3: --- %s minutes ---" % model3_time)

# Score and F1-Score
print("The score is:", clf3_score)
print("The mean F1-Score (unweighted) is:", clf3_f1)

# Show Confusion Matrix
cm3 = pd.crosstab(y_test, y_pred, rownames = ['Realised Class'], colnames = ['Predicted Class'])
display(cm3)

# Show classification report
model3_cr = classification_report(y_test, y_pred)
print(model3_cr)

In [ ]:
# 3. Model 4 - Decision Tree
###
from sklearn.tree import DecisionTreeClassifier
import time
start_time = time.time()

# Instantiate classifier
clf4_name = "Decision Tree"
clf4 = DecisionTreeClassifier(criterion = "entropy", max_depth = 4, random_state = 123)

# Train the model on training data
clf4.fit(X_train, y_train)

# Make predictions on test set
y_pred = clf4.predict(X_test)

# Calc accuracys
clf4_score = clf4.score(X_test, y_test)

# Calc mean unweighted F1 Score in all classes
clf4_f1 = f1_score(y_test, y_pred, average = "macro")

# Measure time
model4_time = (time.time() - start_time)/60


In [ ]:
# Show Results
###

# Modelling Time
print("Model 4: --- %s minutes ---" % model4_time)

# Score and F1-Score
print("The score is:", clf4_score)
print("The mean F1-Score (unweighted) is:", clf4_f1)

# Show Confusion Matrix
cm4 = pd.crosstab(y_test, y_pred, rownames = ['Realised Class'], colnames = ['Predicted Class'])
display(cm4)

# Show classification report
model4_cr = classification_report(y_test, y_pred)
print(model4_cr)
# Ideas: Could show most important Features here. But well, there are 4000 pixels...